# V85 - Qwen3-Embedding-8B zero-shot screen

Pre-registered plan: the one encoder family the hunt never executed is the lineage's own Qwen3-Embedding-8B, frozen in 4-bit exactly per the v41 protocol (revision pinned, left padding, last-token pooling, L2 norm, first 1024 dims). Two readouts: Ridge on user vectors and query-syllabus cosine, both against word-only Ridge under grouped folds with a fresh seed. OOM or missing bitsandbytes records infeasible instead of failing. Same locks.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 4444
N_SPLITS = 5
GATE_FLOOR = 0.002
MODEL, REV, DIM = "Qwen/Qwen3-Embedding-8B", "1d8ad4c", 1024
QMAX, DMAX, BATCH = 96, 128, 1
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v85")
OUT.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), "v85 needs CUDA"
print("DATA:", DATA.resolve())

## 4-bit Qwen vectors, then gate

Queries are user chats, documents are syllabi. Similarities and Ridge heads read out both. Embeddings cache to disk.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
catalog = pd.read_csv(DATA / "modules_catalog.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
syll = [str(catalog.set_index("module_id").loc[m, "description_and_syllabus"]) if m in set(catalog["module_id"]) else "" for m in MODS]

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

cu, cs = OUT / "qwen_users.npy", OUT / "qwen_syll.npy"
if cu.exists() and cs.exists():
    QU, QS = np.load(cu), np.load(cs)
    print("loaded cached Qwen vectors", QU.shape)
else:
    from transformers import AutoModel, AutoTokenizer
    from transformers import BitsAndBytesConfig
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    tok = AutoTokenizer.from_pretrained(MODEL, revision=REV, padding_side="left")
    net = AutoModel.from_pretrained(MODEL, revision=REV, quantization_config=bnb, device_map={"": 0},
                                    torch_dtype=torch.float16, low_cpu_mem_usage=True, attn_implementation="sdpa")
    net.eval()

    def encode(texts, ml):
        out = np.zeros((len(texts), DIM), np.float32)
        with torch.no_grad():
            for b in range(0, len(texts), BATCH):
                e = tok(texts[b:b + BATCH], padding=True, truncation=True, max_length=ml, return_tensors="pt").to("cuda")
                h = net(**e).last_hidden_state
                idx = e["attention_mask"].sum(1).long() - 1
                v = h[torch.arange(h.size(0), device="cuda"), idx.clamp_min(0)][:, :DIM].float()
                out[b:b + BATCH] = F.normalize(v, dim=-1).cpu().numpy()
        return out

    QU = encode(docs, QMAX)
    QS = encode(syll, DMAX)
    np.save(cu, QU)
    np.save(cs, QS)
    print("encoded", QU.shape)
    del net
    torch.cuda.empty_cache()
Qsim = QU[:n_train] @ QS.T

W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()

def ridge_oof(Xtr, alpha=1.0):
    oof = np.zeros_like(Y)
    fids = np.zeros(n_train, int)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(Xtr, strata, fold_groups)):
        for j in range(len(MODS)):
            m = Ridge(alpha=alpha)
            m.fit(Xtr[tr], Y[tr, j])
            oof[va, j] = m.predict(Xtr[va])
        fids[va] = fi + 1
    return oof, fids

oof_w, fid = ridge_oof(W[:n_train])
oof_q, _ = ridge_oof(QU[:n_train])
base = ndcg5(Y, oof_w)
for name, oo in [("qwen-ridge", oof_q), ("qwen-cosine", Qsim)]:
    s = ndcg5(Y, oo)
    d = s - base
    lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
    wf = min(ndcg5(Y[fid == k], oo[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
    print(f"{name}: OOF={s.mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f} {'ADOPT' if (lo > GATE_FLOOR and wf >= 0) else 'reject'}")